In [1]:
import shutil
import os

source = '/kaggle/input/dataset-creation/bnmc_hf_dataset'
destination = '/kaggle/working/bnmc_hf_dataset'

if not os.path.exists(destination):
    shutil.copytree(source, destination)
    print("copied")

copied


In [2]:
from datasets import load_from_disk

ds = load_from_disk('/kaggle/working/bnmc_hf_dataset')


In [3]:
ds['train']

Dataset({
    features: ['text', 'labels'],
    num_rows: 36144
})

In [4]:
from sklearn.preprocessing import MultiLabelBinarizer

def split_labels(example):
    return {'label_list': [l.strip() for l in example['labels'].split(',')]}

ds = ds.map(split_labels)

mlb = MultiLabelBinarizer()
mlb.fit(ds['train']['label_list'])

mlb.classes_

Map:   0%|          | 0/36144 [00:00<?, ? examples/s]

Map:   0%|          | 0/4519 [00:00<?, ? examples/s]

Map:   0%|          | 0/4518 [00:00<?, ? examples/s]

array(['অন্যান্য-খেলা', 'অপরাধ', 'অপহরণ', 'অর্থনীতি', 'আইন',
       'আইন-শৃঙ্খলা', 'আওমীলীগ', 'আন্তর্জাতিক', 'আবহাওয়া', 'আবাসন',
       'উৎসব', 'কূটনীতি', 'কৃষি', 'ক্রিকেট', 'ক্রীড়া', 'জামায়াত',
       'জীবনধারা', 'জেলা সংবাদ', 'ঢাকা', 'দুর্ঘটনা', 'দুর্নীতি',
       'দুর্যোগ', 'ধর্ম', 'নির্বাচন', 'পরিবেশ', 'পর্যটন', 'প্রযুক্তি',
       'ফুটবল', 'বাণিজ্য', 'বি এন পি', 'বিজ্ঞান', 'ব্যাংকিং',
       'যানজট/ট্রাফিক', 'রাজনীতি', 'রোগ-বিস্তার', 'শিক্ষা',
       'শিল্প-কারখানা', 'শেয়ারবাজার', 'সংসদ', 'সংস্কৃতি', 'সমাজ',
       'সরকারি-নীতি', 'সেলিব্রিটি', 'স্বাস্থ্য', 'হত্যা'], dtype=object)

In [5]:
def binarize(batch):
    return {'binarized_labels': mlb.transform(batch['label_list'])}

dataset_binarized = ds.map(binarize, batched = True)
dataset_binarized = dataset_binarized.remove_columns(["labels", "label_list"])
dataset_binarized['test']['binarized_labels'][0]

Map:   0%|          | 0/36144 [00:00<?, ? examples/s]

Map:   0%|          | 0/4519 [00:00<?, ? examples/s]

Map:   0%|          | 0/4518 [00:00<?, ? examples/s]

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("wb_sec")



!huggingface-cli login --token $secret_value_0
!wandb login --relogin $secret_value_1

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `Research stuff` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Research stuff`
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was use

In [7]:
from transformers import AutoTokenizer

model_id = "google/gemma-3-270m-it" 
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

In [8]:
def tokenization(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length =2400 
    )

    tokenized['labels'] = [list(map(float, l)) for l in examples['binarized_labels']]
    return tokenized
    

tokenized_dataset = dataset_binarized.map(
        tokenization, batched = True, remove_columns = dataset_binarized['train'].column_names
    )

Map:   0%|          | 0/36144 [00:00<?, ? examples/s]

Map:   0%|          | 0/4519 [00:00<?, ? examples/s]

Map:   0%|          | 0/4518 [00:00<?, ? examples/s]

In [9]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 36144
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4519
    })
    val: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4518
    })
})

In [10]:
from transformers import AutoModelForSequenceClassification 
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels = len(mlb.classes_),
    problem_type = 'multi_label_classification'
)

config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

2026-02-27 05:20:10.675528: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772169610.880734      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772169610.938054      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772169611.389240      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772169611.389268      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772169611.389271      24 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Some weights of Gemma3TextForSequenceClassification were not initialized from the model checkpoint at google/gemma-3-270m-it and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [12]:
import numpy as np 
from sklearn.metrics import f1_score, roc_auc_score

def compute_metrics(eval_pred):
    init_preds, labels = eval_pred
    probs = 1/(1+np.exp(-init_preds))
    preds = (probs>0.5).astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average= 'micro'),
        'f1_macro': f1_score(labels, preds, average= 'macro'),
        'roc_auc': roc_auc_score(labels, preds, average = 'micro')
    }
    

In [13]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_micro',
    greater_is_better=True,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=10,
    save_total_limit=2,
    learning_rate=2e-5,
    fp16=True,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    report_to = 'wandb'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['val'],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.01)]
)



/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [14]:
trainer.train()

wandb: Currently logged in as: hasnatabdullah47 (uolo) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run hdg32oza
wandb: Tracking run with wandb version 0.22.2
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260227_052031-hdg32oza
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run lemon-feather-22
wandb: ⭐️ View project at https://wandb.ai/uolo/huggingface
wandb: 🚀 View run at https://wandb.ai/uolo/huggingface/runs/hdg32oza
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Roc Auc
1,0.540100,0.207424,0.786641,0.634107,0.880167
2,0.395200,0.212848,0.797200,0.659056,0.893191
3,0.251500,0.222721,0.805501,0.678117,0.901286
4,0.129100,0.266699,0.801598,0.668435,0.900857
5,0.062000,0.299481,0.805145,0.688134,0.905875


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=11295, training_loss=0.2996398471530004, metrics={'train_runtime': 30409.8232, 'train_samples_per_second': 11.886, 'train_steps_per_second': 0.743, 'total_flos': 6.444795837298483e+16, 'train_loss': 0.2996398471530004, 'epoch': 5.0})